# Movie Recommendation System using Matrix Factorisation
**Peyali Chakraborty | M.Sc. Mathematics, Gauhati University**

---

I always wondered how Spotify or Amazon knows what I'll like before I've even tried it. The answer is Matrix Factorisation — a linear algebra technique that predicts missing ratings from patterns in existing ones.

## 1. The Math

Given a ratings matrix **R** (m users × n movies) with most entries missing, I want to find:
- **P** ∈ ℝ^(m×k) — user latent factors
- **Q** ∈ ℝ^(n×k) — movie latent factors

such that $\hat{R} = P \cdot Q^T$

I chose k=3 latent dimensions. The model learns what these dimensions represent on its own.

**Loss** (only over observed entries, with L2 regularisation):
$$\mathcal{L} = \sum_{(i,j) \in \Omega} \left(R_{ij} - P_i \cdot Q_j^T\right)^2 + \lambda \left(\|P\|_F^2 + \|Q\|_F^2\right)$$

**Gradient updates** — for error $e_{ij} = R_{ij} - P_i \cdot Q_j^T$:
$$P_i \leftarrow P_i + \alpha (e_{ij} \cdot Q_j - \lambda P_i)$$
$$Q_j \leftarrow Q_j + \alpha (e_{ij} \cdot P_i - \lambda Q_j)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 2. Ratings Matrix

6 users × 6 movies. I kept it small so I can see exactly what the algorithm does at each step. `NaN` = not watched.

In [ ]:
movies = ["3 Idiots", "Drishyam 2", "Phir Hera Pheri", "Welcome", "Baby's Day Out", "Hotel Transylvania"]
users  = ["Peyali", "Rahul", "Sneha", "Arjun", "Priya", "Karan"]

R_raw = np.array([
#  3Idiots  Drishyam2  PHF    Welcome  BabyDayOut  HotelT
   [5,       4,        5,     5,        4,          np.nan],  # Peyali
   [4,       5,        np.nan,4,        np.nan,     3     ],  # Rahul
   [np.nan,  3,        4,     np.nan,   5,          5     ],  # Sneha
   [3,       np.nan,   5,     4,        np.nan,     4     ],  # Arjun
   [5,       4,        np.nan,np.nan,   4,          5     ],  # Priya
   [np.nan,  3,        3,     5,        5,          np.nan],  # Karan
])

m, n = R_raw.shape
print(f"{int(np.sum(~np.isnan(R_raw)))} observed ratings, {int(np.sum(np.isnan(R_raw)))} missing")

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

def plot_matrix(ax, matrix, title):
    cmap_base = plt.cm.get_cmap('Blues', 5)
    colors = ['#d0d0d0'] + [cmap_base(i) for i in range(5)]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm([-1.5,0.5,1.5,2.5,3.5,4.5,5.5], cmap.N)
    display = np.where(np.isnan(matrix), -1, matrix)
    im = ax.imshow(display, cmap=cmap, norm=norm, aspect='auto')
    for i in range(m):
        for j in range(n):
            val = matrix[i, j]
            txt = '?' if np.isnan(val) else str(int(val))
            color = '#888' if np.isnan(val) else ('white' if val >= 4 else 'black')
            ax.text(j, i, txt, ha='center', va='center', fontsize=13, fontweight='bold', color=color)
    ax.set_xticks(range(n)); ax.set_xticklabels(movies, rotation=25, ha='right', fontsize=9)
    ax.set_yticks(range(m)); ax.set_yticklabels(users, fontsize=9)
    ax.set_title(title, fontsize=11, pad=10)
    return im

fig, ax = plt.subplots(figsize=(9, 4))
im = plot_matrix(ax, R_raw, 'Ratings Matrix  (? = not watched)')
plt.colorbar(im, ax=ax, ticks=[1,2,3,4,5])
plt.tight_layout()
plt.show()

## 3. Matrix Factorisation

In [ ]:
def matrix_factorisation(R, k=3, alpha=0.005, lam=0.02, n_epochs=3000):
    m, n = R.shape
    P = np.random.normal(0, 0.1, (m, k))
    Q = np.random.normal(0, 0.1, (n, k))
    observed = [(i, j) for i in range(m) for j in range(n) if not np.isnan(R[i, j])]
    losses = []

    for epoch in range(n_epochs):
        np.random.shuffle(observed)
        for (i, j) in observed:
            e_ij = R[i, j] - P[i] @ Q[j]
            P[i] += alpha * (e_ij * Q[j] - lam * P[i])
            Q[j] += alpha * (e_ij * P[i] - lam * Q[j])
        rmse = np.sqrt(np.mean([(R[i,j] - P[i] @ Q[j])**2 for (i,j) in observed]))
        losses.append(rmse)

    return P, Q, losses


P, Q, losses = matrix_factorisation(R_raw)
R_pred = np.clip(P @ Q.T, 1, 5)

print(f"Final RMSE: {losses[-1]:.4f}")

## 4. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(losses, color='#2563eb', linewidth=1.5)
ax.axhline(losses[-1], color='red', linestyle='--', alpha=0.6, label=f'Final RMSE = {losses[-1]:.4f}')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.set_title('Gradient Descent — Reconstruction Error')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Original vs Reconstructed

Orange borders = entries that were missing and are now predicted.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

im = plot_matrix(axes[0], R_raw, 'Original')
plt.colorbar(im, ax=axes[0], ticks=[1,2,3,4,5])

im2 = axes[1].imshow(R_pred, cmap='Blues', vmin=1, vmax=5, aspect='auto')
for i in range(m):
    for j in range(n):
        val = R_pred[i, j]
        color = 'white' if val >= 4 else 'black'
        axes[1].text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=11, fontweight='bold', color=color)
        if np.isnan(R_raw[i, j]):
            rect = plt.Rectangle((j-0.48, i-0.48), 0.96, 0.96, fill=False, edgecolor='#f59e0b', linewidth=2)
            axes[1].add_patch(rect)
axes[1].set_xticks(range(n)); axes[1].set_xticklabels(movies, rotation=25, ha='right', fontsize=9)
axes[1].set_yticks(range(m)); axes[1].set_yticklabels(users, fontsize=9)
axes[1].set_title('Reconstructed  (orange = predicted)', fontsize=11, pad=10)
plt.colorbar(im2, ax=axes[1], ticks=[1,2,3,4,5])

plt.tight_layout()
plt.show()

## 6. Top-3 Recommendations

In [ ]:
for i, user in enumerate(users):
    unseen = [j for j in range(n) if np.isnan(R_raw[i, j])]
    ranked = sorted(unseen, key=lambda j: R_pred[i, j], reverse=True)[:3]
    print(f"{user}:")
    for rank, j in enumerate(ranked, 1):
        print(f"  {rank}. {movies[j]:<22} {R_pred[i,j]:.2f}")
    print()

In [ ]:
i = 0
unseen_j = [j for j in range(n) if np.isnan(R_raw[i, j])]
pred_vals = [R_pred[i, j] for j in unseen_j]
unseen_movies = [movies[j] for j in unseen_j]

fig, ax = plt.subplots(figsize=(7, 3))
colors_bar = ['#2563eb' if v == max(pred_vals) else '#93c5fd' for v in pred_vals]
bars = ax.barh(unseen_movies, pred_vals, color=colors_bar, height=0.5)
for bar, val in zip(bars, pred_vals):
    ax.text(val + 0.08, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 5.5)
ax.set_xlabel('Predicted Rating')
ax.set_title(f'Recommendations for {users[i]}')
plt.tight_layout()
plt.show()

## 7. User Similarity in Latent Space

Each user is a vector in k-dimensional space (their row in P). Cosine similarity between these vectors shows how similar two users' tastes are.

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

sim = np.array([[cosine_sim(P[i], P[j]) for j in range(m)] for i in range(m)])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=users, yticklabels=users,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('User–User Cosine Similarity')
plt.tight_layout()
plt.show()